# ⚖️ Página 2 del BI — Comparación A vs B

Replica la segunda página del BI oficial: selectores jerárquicos (GITT / PRB / Regional), dos gráficas de barras espejadas y los labels analíticos finales (Hallazgo Principal, Resumen Desempeño, Estados y Brecha).

Usamos el mismo helper `loaders.py` de Página 1 para cargar desde Excel o DB indistintamente.

## 1. Imports y carga de datos

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "src"))
from loaders import load_from_excel, load_from_db, describe_dict, compare_sources

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

In [ ]:
# Excel
data_excel = load_from_excel(Path("data/Metas 2026_GITT_ Para power BI.xlsx"))

# DB (API)
try:
    data_db = load_from_db("http://localhost/api", "admin", "Admin@UBPD2024!")
    print("✓ Ambas fuentes cargadas")
except Exception as exc:
    print(f"⚠️  Solo Excel disponible: {exc}")
    data_db = None

## 2. Descripción y comparación rápida

In [ ]:
if data_db is not None:
    compare_sources(data_excel, data_db)

## 3. Elegir fuente activa

Con la misma convención que el Notebook 1 — cambia `SOURCE` para alternar.

In [ ]:
SOURCE = "excel"   # "excel" | "db"

data = data_excel if SOURCE == "excel" else data_db
print(f"🔵 Fuente activa: {SOURCE}")

## 4. Preparar tablas

Limpieza idéntica a la Página 1 + merge con `PRB` para tener `Regional` y `GITT` en cada fila.

In [ ]:
hist = data["Historico"].copy()
prb  = data["PRB"].copy()

# Limpieza
hist = hist.dropna(subset=["CodPRB", "Cod_Indicador", "AÑO"])
hist = hist[(hist["CodPRB"] != 0) & (hist["Cod_Indicador"] != 0)]
for col in ["Meta", "Avance Total", "Línea Base 2025"]:
    if col in hist.columns:
        hist[col] = pd.to_numeric(hist[col], errors="coerce").fillna(0)

# Merge para tener Regional/GITT/PRB_nombre
enriched = hist.merge(
    prb[["COD", "PRB", "Regional", "GITT"]].rename(columns={"PRB": "PRB_nombre"}),
    left_on="CodPRB", right_on="COD", how="left",
)
enriched.head(3)

## 5. Selectores jerárquicos disponibles

El BI ofrece un dropdown en árbol con 3 niveles:
- **GITT** → opciones: `ANTIOQUIA`, `ARAUCA`, `ATLÁNTICO`…
- **PRB**  → opciones: `Alta Y Media Guajira`, `Alto Putumayo`…
- **Regional** → opciones: `CENTRO`, `NOROCCIDENTE`, `NORORIENTE`…

In [ ]:
niveles = {
    "gitt":     sorted(prb["GITT"].dropna().unique().tolist()),
    "prb":      sorted(prb["PRB"].dropna().unique().tolist()),
    "regional": sorted(prb["Regional"].dropna().unique().tolist()),
}

for nivel, opciones in niveles.items():
    print(f"{nivel.upper():10} → {len(opciones)} opciones. Primeras 5: {opciones[:5]}")

## 6. Función de fetch por grupo

Dado un `(nivel, valor)` retorna por cada indicador su `Meta` y `Avance`. Es la misma lógica que implementa el backend en `/api/bi/comparison`.

In [ ]:
def fetch_grupo(df: pd.DataFrame, nivel: str, valor: str | None, anio: int = 2026) -> pd.DataFrame:
    """
    Filtra el DataFrame enriched por (nivel, valor) y agrupa por indicador.
    Devuelve columnas: codigo, nombre, meta, avance.
    """
    sub = df[df["AÑO"] == anio]
    if nivel == "gitt" and valor:
        sub = sub[sub["GITT"] == valor]
    elif nivel == "regional" and valor:
        sub = sub[sub["Regional"] == valor]
    elif nivel == "prb" and valor:
        sub = sub[sub["PRB_nombre"] == valor]
    elif nivel == "meta":
        pass   # sin filtro (usa TODOS los datos = metas globales)

    agg = (
        sub.groupby(["Código del Indicador", "Indicador"], as_index=False)
           .agg(meta=("Meta", "sum"), avance=("Avance Total", "sum"))
           .sort_values("Código del Indicador")
           .reset_index(drop=True)
    )
    return agg

## 7. Selección A y B

Cambia `SEL_A` y `SEL_B` para comparar distintos pares. Por ejemplo:
- `("gitt", "ANTIOQUIA")` vs `("gitt", "META")` (global)
- `("regional", "CENTRO")` vs `("regional", "NOROCCIDENTE")`

In [ ]:
# (nivel, valor)  — valor=None si nivel=="meta"
SEL_A = ("gitt",     "ANTIOQUIA")
SEL_B = ("regional", "NOROCCIDENTE")

grupo_a = fetch_grupo(enriched, SEL_A[0], SEL_A[1])
grupo_b = fetch_grupo(enriched, SEL_B[0], SEL_B[1])

print(f"Grupo A: {SEL_A}  →  {len(grupo_a)} indicadores")
print(f"Grupo B: {SEL_B}  →  {len(grupo_b)} indicadores")

In [ ]:
grupo_a

In [ ]:
grupo_b

## 8. Unificar en un solo DataFrame para graficar

Por cada indicador necesitamos `a` (avance del grupo A) y `b` (avance del grupo B).

In [ ]:
merged = pd.merge(
    grupo_a[["Código del Indicador", "Indicador", "avance"]].rename(columns={"avance": "a"}),
    grupo_b[["Código del Indicador", "avance"]].rename(columns={"avance": "b"}),
    on="Código del Indicador",
    how="outer",
).fillna(0)
merged = merged.sort_values("Código del Indicador").reset_index(drop=True)
merged

## 9. Dos gráficas de barras espejadas

Igual que el BI: A a la izquierda (crece hacia la izquierda), B a la derecha.

In [ ]:
# Nombres cortos para el eje Y
SHORT_LABELS = {
    "L1P-002":     "PDD con solicitud de búsqueda",
    "L1A-021":     "SB Mejoradas Pendientes",
    "L1A-020a":    "PDD con muestra biológica asociada",
    "L1P-010":     "No. de lugares de IF caracterizados",
    "L1R-006-007": "SIF (confirmados y descartados)",
    "L1R-001":     "PEV PRB Asignado",
    "L1R-005":     "Entrega Digna GITT asignada PDD",
    "L1A-022":     "Postulados Búsq Inversa para Verif.",
    "L1R-004":     "Personas con contacto exitoso",
    "L1R-008":     "Cuerpos Recuperados",
    "L1P-006":     "Planes de trabajo con aportantes",
    "L1P-008-009": "Informe Investigación con Hipótesis",
    "L1R-003":     "Informes de lo acaecido entregados",
}

labels_plot = [SHORT_LABELS.get(c, c) for c in merged["Código del Indicador"]]

maxabs = max(merged["a"].abs().max(), merged["b"].abs().max(), 1)
y = np.arange(len(merged))

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(14, 7), sharey=True,
                                  gridspec_kw={"wspace": 0.35})

# Grupo A — eje X invertido
ax_a.barh(y, merged["a"], color="#9B5F7E", height=0.5)
ax_a.set_xlim(maxabs, 0)                # invertido
ax_a.set_yticks(y)
ax_a.set_yticklabels(labels_plot)
ax_a.yaxis.tick_right()
ax_a.set_title(f"Grupo A — {SEL_A[0].upper()} | {SEL_A[1]}", color="#7C3A5C", fontweight="bold")
ax_a.spines[["top", "right"]].set_visible(False)
for i, v in enumerate(merged["a"]):
    if v > 0:
        ax_a.text(v, i, f" {v:,.0f}", va="center", ha="left", fontsize=9, color="#555")

# Grupo B
ax_b.barh(y, merged["b"], color="#35678A", height=0.5)
ax_b.set_xlim(0, maxabs)
ax_b.set_title(f"Grupo B — {SEL_B[0].upper()} | {SEL_B[1]}", color="#35678A", fontweight="bold")
ax_b.spines[["top", "right", "left"]].set_visible(False)
ax_b.tick_params(left=False, labelleft=False)
for i, v in enumerate(merged["b"]):
    if v > 0:
        ax_b.text(v, i, f" {v:,.0f}", va="center", ha="left", fontsize=9, color="#555")

plt.suptitle("Comparación A vs B — Página 2 del BI", fontsize=14, fontweight="bold", y=1.00)
plt.tight_layout()
plt.show()

## 10. Labels analíticos de la parte inferior

En el BI aparecen 5 cuadros:

| # | Cuadro | Fórmula |
|---|---|---|
| 1 | Hallazgo Principal | Indicador con mayor gap absoluto entre A y B, con la ventaja expresada como % |
| 2 | Resumen Desempeño | Comparación narrativa del % de ejecución de cada grupo |
| 3 | Estado Selección Filtro A | 🟢/🟡/🟠/🔴 según % de ejecución |
| 4 | Estado Selección Filtro B | idem |
| 5 | Brecha A vs B | Diferencia absoluta con indicación de quién supera |

In [ ]:
total_avance_a = merged["a"].sum()
total_avance_b = merged["b"].sum()
total_meta_a   = grupo_a["meta"].sum()
total_meta_b   = grupo_b["meta"].sum()

pct_a = (total_avance_a / total_meta_a * 100) if total_meta_a else 0
pct_b = (total_avance_b / total_meta_b * 100) if total_meta_b else 0

print(f"A: avance={total_avance_a:,.0f}  meta={total_meta_a:,.0f}  %={pct_a:.2f}")
print(f"B: avance={total_avance_b:,.0f}  meta={total_meta_b:,.0f}  %={pct_b:.2f}")

In [ ]:
# 1) Hallazgo Principal — mayor gap absoluto
merged["gap"] = merged["a"] - merged["b"]
idx_max = merged["gap"].abs().idxmax()
gap_max = merged.loc[idx_max, "gap"]
ind_max = merged.loc[idx_max, "Indicador"]
ventaja = "A supera a B" if gap_max > 0 else "B supera a A"
denominador = max(abs(total_avance_a), abs(total_avance_b), 1)
pct_gap = abs(gap_max) / denominador * 100
hallazgo = f"La mayor diferencia se observa en el indicador '{ind_max}', donde {ventaja} en {pct_gap:.1f}%."

# 2) Resumen Desempeño
resumen = (
    f"Grupo A alcanza {pct_a:.1f} % de su meta y Grupo B alcanza {pct_b:.1f} %. "
    f"Diferencia absoluta de avance: {abs(total_avance_a - total_avance_b):,.0f} unidades."
)

# 3, 4) Estado (tier por % ejecución)
def tier(pct: float) -> str:
    if pct >= 70:  return "🟢 Óptimo"
    if pct >= 40:  return "🟡 Moderado"
    if pct  > 0:   return "🟠 En progreso"
    return "🔴 Crítico"

estado_a = f"{SEL_A[0].upper()} | {SEL_A[1]}  →  {tier(pct_a)}  ({pct_a:.1f} %)"
estado_b = f"{SEL_B[0].upper()} | {SEL_B[1]}  →  {tier(pct_b)}  ({pct_b:.1f} %)"

# 5) Brecha A vs B
brecha_val = total_avance_a - total_avance_b
if   brecha_val > 0:   brecha = f"A supera a B en {brecha_val:,.0f} unidades de avance."
elif brecha_val < 0:   brecha = f"B supera a A en {abs(brecha_val):,.0f} unidades de avance."
else:                  brecha = "A y B tienen el mismo avance total."

print("─" * 70)
print("HALLAZGO PRINCIPAL");             print(f"  {hallazgo}")
print("─" * 70)
print("RESUMEN DESEMPEÑO");              print(f"  {resumen}")
print("─" * 70)
print("ESTADO SELECCIÓN FILTRO A");      print(f"  {estado_a}")
print("ESTADO SELECCIÓN FILTRO B");      print(f"  {estado_b}")
print("─" * 70)
print("BRECHA A vs B");                  print(f"  {brecha}")
print("─" * 70)

## 11. Contraste contra el endpoint del backend

El backend UBPD expone esta misma comparación en `GET /api/bi/comparison`. Contrastamos que los números coincidan.

In [ ]:
import requests

params = {
    "a_tipo":  SEL_A[0], "a_valor": SEL_A[1],
    "b_tipo":  SEL_B[0], "b_valor": SEL_B[1],
    "anio":    2026,
}
try:
    r = requests.get("http://localhost/api/bi/comparison", params=params, timeout=30)
    r.raise_for_status()
    api_resp = r.json()

    api_cmp = pd.DataFrame(api_resp["indicadores"])
    contraste = merged.merge(
        api_cmp.rename(columns={"codigo": "Código del Indicador",
                                 "a": "api_a", "b": "api_b"}),
        on="Código del Indicador", how="outer",
    )
    contraste["dif_a"] = contraste["a"] - contraste["api_a"]
    contraste["dif_b"] = contraste["b"] - contraste["api_b"]
    display_cols = ["Código del Indicador", "a", "api_a", "dif_a", "b", "api_b", "dif_b"]
    contraste[display_cols]
except Exception as exc:
    print(f"⚠️  No se pudo contrastar contra la API: {exc}")

✅ Si `dif_a` y `dif_b` son todos **0**, significa que nuestro cálculo en pandas coincide exactamente con el que hace el backend al alimentar el BI.